# Managing Message State & History in LangChain
This notebook covers how to use LangChain's message schemas (`SystemMessage`, `HumanMessage`, `AIMessage`, and `ToolMessage`) to direct model behavior and build multi-turn chat applications.

### Step 1: Initialize Chat Model

In [ ]:
import os
from langchain_groq import ChatGroq

# Ensure the API key is in the environment
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Initialize model
groq_model = ChatGroq(model="qwen/qwen3-32b")

### Step 2: Simple String Invocation
Under the hood, passing a string is automatically wrapped in a `HumanMessage`.

In [ ]:
response = groq_model.invoke("Tell me about the rules of badminton game.")
print(response.content[:500] + "...")

### Step 3: Structured Messages (System & Human Messages)
We use `SystemMessage` to set instructions or context for the assistant, and `HumanMessage` for user queries.

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# Define the messages array
messages = [
    SystemMessage("You are a poetry expert."),
    HumanMessage("Write a poem on the name TEJA .")
]

# Get the response from model
response = groq_model.invoke(messages)
response.content

### Step 4: Multi-Turn Conversation History
By passing previous `AIMessage` responses back with new user inputs, the model maintains full context of the conversation.

In [ ]:
ai_msg = AIMessage("I'd be happy to help you with that.")

# Build conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,
    HumanMessage("Great! What is the factorial of 5 ?")
]

# Get final response
response = groq_model.invoke(messages)
print(response.content)

### Step 5: Check Usage Metadata (Tokens)

In [ ]:
# Inspect input/output token counts
response.usage_metadata

### Step 6: Managing History Limits (Context Truncation / Pruning)
As conversation histories grow, they can exceed context window limits or increase API costs. A common practice is trimming the history before invoking the model while keeping the main instructions (`SystemMessage`) intact.

In [ ]:
def trim_history(messages_list, max_turns=2):
    """
    Keeps the SystemMessage(s) at the start, and retains only 
    the last `max_turns * 2` messages of conversational interaction.
    """
    system_messages = [msg for msg in messages_list if isinstance(msg, SystemMessage)]
    chat_messages = [msg for msg in messages_list if not isinstance(msg, SystemMessage)]
    
    # Keep only the last N chat messages (human/AI alternates)
    trimmed_chat = chat_messages[-(max_turns * 2):]
    
    return system_messages + trimmed_chat

# Create a simulated long conversation
full_chat = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("My favorite color is green."),
    AIMessage("That's a nice color!"),
    HumanMessage("I live in Seattle."),
    AIMessage("Cool! Seattle is a beautiful city."),
    HumanMessage("What is my favorite color?")
]

print(f"Original Chat Length: {len(full_chat)} messages.")

# Trim to only keep the last 1 turn (last 2 messages of chat) + the system prompt
trimmed_chat = trim_history(full_chat, max_turns=1)
print(f"Trimmed Chat Length: {len(trimmed_chat)} messages.")
print("\nTrimmed Chat Context:")
for msg in trimmed_chat:
    print(f"[{type(msg).__name__}]: {msg.content}")

### Step 7: Complete Message flow in Tool Calling (Visualization)
Here we show a complete sequence of LangChain message states that occur during a single tool call request and response loop.

In [ ]:
# A mock sequence representing how LangGraph or LangChain executes a tool-calling flow:
tool_flow_messages = [
    SystemMessage("You are an assistant with access to weather tools."),
    
    # 1. User message
    HumanMessage("Is it raining in London?"),
    
    # 2. Assistant message indicating a tool call is needed
    AIMessage(
        content="", 
        tool_calls=[{
            "name": "get_weather", 
            "args": {"location": "London"}, 
            "id": "call_ldn_123", 
            "type": "tool_call"
        }]
    ),
    
    # 3. Tool response message containing the outcome of the Python function execution
    ToolMessage(
        content="It is currently raining heavily in London with a temperature of 14°C.",
        name="get_weather",
        tool_call_id="call_ldn_123"
    )
]

print("Pre-constructed messages for tool-use verification:")
for idx, msg in enumerate(tool_flow_messages):
    print(f"Step {idx}: [{type(msg).__name__}] -> Content: '{msg.content}'")